In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim, to_date

df_careplans_bronze = spark.read.table("workspace.bronze.careplans")

id_col = "id" if "id" in df_careplans_bronze.columns else "ID"

df_careplans_silver = (
    df_careplans_bronze
    .filter(col(id_col).isNotNull())
    .dropDuplicates([id_col])
    .select(
        col(id_col).alias("careplan_id"),
        to_date(col("start"), "yyyy-MM-dd").alias("start_date"),
        to_date(col("stop"), "yyyy-MM-dd").alias("stop_date"),
        col("patient").alias("patient_id"),
        col("encounter").alias("encounter_id"),
        col("code").alias("careplan_code"),
        trim(col("description")).alias("careplan_description"),
        col("reasoncode").alias("reason_code"),
        trim(col("reasondescription")).alias("reason_description"),
        col("ingested_at")
    )
)

(
    df_careplans_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.careplans")
)

print(f"✅ Created workspace.silver.careplans with {df_careplans_silver.count()} clean rows!")